
# Лабораторная работа
## «Разложение периодических сигналов в ряд Фурье и спектральный анализ»

**Курс:** Радиотехнические цепи и сигналы (базовый уровень)
**Цель работы:** изучить физический смысл разложения периодического сигнала в ряд Фурье; научиться строить амплитудный спектр; экспериментально проверить связь формы сигнала (скважность, постоянная составляющая) с его спектром в Python и NI Multisim.

**Программное обеспечение:**
- Python 3 (библиотеки `numpy`, `matplotlib`);
- NI Multisim (виртуальные приборы: функциональный генератор `XFG1`, осциллограф `XSC1`, анализатор спектра `XSA1`).

**Порядок выполнения:** ноутбук разбит на 4 шага. В каждом шаге сначала читаете теорию, затем выполняете код, затем повторяете опыт в Multisim и заполняете таблицу сравнения. В конце — таблица вариантов и контрольные вопросы.



---
## Шаг 1. Генерация периодического сигнала (прямоугольные импульсы)

### Теория (кратко)

Периодическим называется сигнал, значения которого повторяются через период $T$:
$$s(t) = s(t + T).$$

Простейший периодический сигнал в радиотехнике — **последовательность прямоугольных импульсов** (меандр при скважности 2). Он характеризуется:
- амплитудой $A$ (В);
- периодом $T$ (с) или частотой $f_1 = 1/T$ (Гц);
- длительностью импульса $\tau$ (с);
- **скважностью** $q = T/\tau$ (безразмерная величина).

Временная форма такого сигнала:
$$s(t) =
\begin{cases}
A, & 0 \le t < \tau,\\
0, & \tau \le t < T.
\end{cases}$$

### Что вы сейчас сделаете
1. Зададите параметры сигнала (свои по варианту — см. таблицу в конце).
2. Сформируете массив времени и посчитаете значения сигнала «вручную» (без готовых функций).
3. Построите график $s(t)$ с подписанными осями.

**Ожидаемый результат:** на экране появится «лесенка» из прямоугольных импульсов с амплитудой $A$ и периодом $T$. Убедитесь, что длительность импульса соответствует $\tau$.


In [ ]:

# ============================================================
# ШАГ 1. ГЕНЕРАЦИЯ ПРЯМОУГОЛЬНОГО ПЕРИОДИЧЕСКОГО СИГНАЛА
# ============================================================
import numpy as np                      # numpy — массивы и математика
import matplotlib.pyplot as plt         # matplotlib — построение графиков

# ---------- ПАРАМЕТРЫ СИГНАЛА (ВПИШИТЕ СВОИ ЗНАЧЕНИЯ ПО ВАРИАНТУ) ----------
A = 5.0          # <--- ВПИШИТЕ СВОЕ ЗНАЧЕНИЕ ЗДЕСЬ: амплитуда, В
f1 = 1000.0      # <--- ВПИШИТЕ СВОЕ ЗНАЧЕНИЕ ЗДЕСЬ: частота, Гц
tau = 0.0005     # <--- ВПИШИТЕ СВОЕ ЗНАЧЕНИЕ ЗДЕСЬ: длительность импульса, с

T = 1.0 / f1     # период сигнала, с (формула T = 1/f)
q = T / tau      # скважность q = T/tau (безразмерная)

# ---------- ОСЬ ВРЕМЕНИ ----------
# Показываем 3 периода сигнала. Шаг дискретизации dt = T/1000,
# чтобы на графике было достаточно точек (1000 точек на период).
t = np.linspace(0.0, 3.0 * T, 3000)     # массив времени, с

# ---------- ФОРМИРОВАНИЕ СИГНАЛА ----------
# np.mod(t, T) — остаток от деления времени на период: «фаза внутри периода».
# Если остаток меньше tau — импульс есть (A), иначе — ноль.
s = np.where(np.mod(t, T) < tau, A, 0.0)   # значения сигнала, В

# ---------- ПОСТРОЕНИЕ ГРАФИКА ----------
plt.figure(figsize=(10, 4))                       # размер рисунка
plt.plot(t * 1000, s, color='blue', linewidth=2, label='s(t)')  # t в мс
plt.title(f'Прямоугольный сигнал: A={A} В, f1={f1} Гц, τ={tau*1e6:.0f} мкс, q={q:.2f}')
plt.xlabel('Время t, мс')                        # подпись оси X с единицами
plt.ylabel('Напряжение s(t), В')                  # подпись оси Y с единицами
plt.grid(True, which='both', linestyle='--', alpha=0.7)  # сетка
plt.legend(loc='upper right')                     # легенда
plt.xlim(0, 3 * T * 1000)                         # диапазон по X в мс
plt.tight_layout()                                # аккуратные отступы
plt.show()                                        # показать график



### Параллельный эксперимент в NI Multisim (Шаг 1)

**Цель:** увидеть ту же «лесенку» на виртуальном осциллографе.

**Что взять на схеме:**
| Компонент | Обозначение в Multisim | Куда подключить |
|---|---|---|
| Функциональный генератор | `XFG1` | Выход `+` → вход канала A осциллографа |
| Осциллограф | `XSC1` | Канал A: `+` — сигнал, `-` — земля |
| Земля | `GROUND` | Общая точка генератора и осциллографа |

**Пошагово:**
1. В библиотеке `Sources` найдите `FUNCTION_GENERATOR` (или `XFG1`). Поставьте на поле.
2. В окне свойств `XFG1` выберите **форму «Прямоугольник»** (иконка $\sqcap$).
3. Задайте **частоту** $f_1$ (например, 1 кГц), **амплитуду** $A$ (5 В), **скважность** (в Multisim задаётся параметр **Duty Cycle** = $\tau/T \cdot 100\%$; для скважности $q=2$ это 50 %).
4. Подключите `XFG1` к каналу A `XSC1`, второй вывод — к земле.
5. Нажмите **Run** (зелёный треугольник). Откройте двойным щелчком `XSC1`, отрегулируйте **Timebase** (например, 1 мс/дел) и **Volts/Div** (2 В/дел), чтобы увидеть 2–3 периода.
6. Зарисуйте форму или сделайте скриншот.

### Таблица сравнения результатов Шага 1

| Параметр | Теория (формула) | Python | Multisim | Вывод |
|---|---|---|---|---|
| Амплитуда $A$, В | | | | |
| Период $T$, мс | | | | |
| Длительность $\tau$, мкс | | | | |
| Скважность $q$ | | | | |



---
## Шаг 2. Синтез сигнала из гармоник (ряд Фурье)

### Теория (кратко)

Любой периодический сигнал $s(t)$ с периодом $T$ можно представить суммой гармоник (ряд Фурье):
$$s(t) = \frac{a_0}{2} + \sum_{n=1}^{\infty} \left[ a_n \cos(2\pi n f_1 t) + b_n \sin(2\pi n f_1 t) \right], \quad f_1 = \frac{1}{T}.$$

Для **прямоугольной последовательности** с амплитудой $A$ и скважностью $q$ коэффициенты имеют вид:
$$a_0 = \frac{2A}{q}, \qquad a_n = \frac{2A}{n\pi}\sin\left(\frac{n\pi}{q}\right), \qquad b_n = 0.$$

Тогда ряд записывается проще:
$$s(t) = \frac{A}{q} + \sum_{n=1}^{\infty} \frac{2A}{n\pi}\sin\left(\frac{n\pi}{q}\right)\cos(2\pi n f_1 t).$$

**Физический смысл:** чем больше гармоник $N$ мы суммируем, тем ближе сумма к «идеальному» прямоугольнику. Одиночная синусоида — это только первая гармоника; добавление 3-й, 5-й и т.д. «заостряет» фронты (появляется эффект Гиббса — выбросы у фронтов).

### Что вы сейчас сделаете
1. Зададите число гармоник $N$ (начните с $N = 1$, затем $N = 5$, $N = 20$).
2. Посчитаете сумму по формуле выше (цикл `for`).
3. Сравните синтезированный сигнал с идеальным прямоугольником на одном графике.

**Ожидаемый результат:** при $N=1$ — почти синусоида; при $N=20$ — почти прямоугольник с небольшими пульсациями у фронтов.


In [ ]:

# ============================================================
# ШАГ 2. СИНТЕЗ СИГНАЛА ИЗ ГАРМОНИК (РЯД ФУРЬЕ)
# ============================================================
# Параметры A, f1, tau, T, q и массив t уже заданы в Шаге 1.
# Если вы перезапускали ядро — выполните сначала ячейку Шага 1.

N = 20           # <--- ВПИШИТЕ СВОЕ ЗНАЧЕНИЕ ЗДЕСЬ: число гармоник N

# ---------- ПОСТОЯННАЯ СОСТАВЛЯЮЩАЯ ----------
a0 = 2.0 * A / q                 # a0 = 2A/q (для нашего ряда)
s_synth = np.full_like(t, a0 / 2.0)   # начинаем сумму с a0/2 (постоянная часть)

# ---------- СУММИРОВАНИЕ ГАРМОНИК ----------
for n in range(1, N + 1):                       # цикл по номерам гармоник 1..N
    an = (2.0 * A / (n * np.pi)) * np.sin(n * np.pi / q)  # коэффициент a_n
    # гармоника: a_n * cos(2*pi*n*f1*t)
    s_synth += an * np.cos(2.0 * np.pi * n * f1 * t)      # добавляем к сумме

# ---------- ИДЕАЛЬНЫЙ ПРЯМОУГОЛЬНИК ДЛЯ СРАВНЕНИЯ ----------
s_ideal = np.where(np.mod(t, T) < tau, A, 0.0)  # та же формула, что в Шаге 1

# ---------- ГРАФИК ----------
plt.figure(figsize=(10, 4))
plt.plot(t * 1000, s_ideal, 'k--', linewidth=1.5, label='Идеальный прямоугольник')
plt.plot(t * 1000, s_synth, 'r', linewidth=2, label=f'Сумма {N} гармоник')
plt.title(f'Синтез прямоугольного сигнала рядом Фурье (N = {N})')
plt.xlabel('Время t, мс')
plt.ylabel('Напряжение s(t), В')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(loc='upper right')
plt.xlim(0, 3 * T * 1000)
plt.tight_layout()
plt.show()

# ---------- ЧИСЛОВАЯ ОЦЕНКА ОШИБКИ ----------
# Среднеквадратичное отклонение синтеза от идеала (чем меньше — тем лучше)
error = np.sqrt(np.mean((s_synth - s_ideal) ** 2))
print(f'СКО синтеза и идеала при N={N}: {error:.4f} В')



### Параллельный эксперимент в NI Multisim (Шаг 2)

**Цель:** собрать полигармонический сигнал из нескольких источников и убедиться, что сумма синусоид даёт «ступенчатый» прямоугольник.

**Что взять на схеме:**
| Компонент | Обозначение | Параметры |
|---|---|---|
| Источник синусоиды №1 | `AC_VOLTAGE` (V1) | $f = f_1$, $A_1 = 2A/\pi$ (для $q=2$) |
| Источник синусоиды №2 | `AC_VOLTAGE` (V2) | $f = 3f_1$, $A_3 = 2A/(3\pi)$ |
| Источник синусоиды №3 | `AC_VOLTAGE` (V3) | $f = 5f_1$, $A_5 = 2A/(5\pi)$ |
| Сумматор | `SUMMER` (или соедините последовательно) | — |
| Осциллограф | `XSC1` | канал A — сумма |

> 💡 В Multisim проще всего соединить источники **последовательно** (их напряжения складываются) и подать общую точку на осциллограф. Либо использовать компонент `SUMMER` из группы `Analog`.

**Пошагово:**
1. Поставьте три `AC_VOLTAGE`. В свойствах каждого задайте частоту и амплитуду (амплитуда в Multisim — это **амплитудное** значение, не действующее!).
2. Соедините их последовательно (плюс одного к минусу другого), крайние выводы — на `XSC1`.
3. Запустите **Run**, настройте развёртку так, чтобы видеть 2–3 периода $T$.
4. Сравните форму с результатом Python при $N = 3$. Затем добавьте 5-ю и 7-ю гармоники (ещё источники) — фронты станут круче.

### Таблица сравнения результатов Шага 2

| Число гармоник $N$ | Форма в Python (синус/прямоугольник) | Форма в Multisim | СКО (Python), В | Вывод |
|---|---|---|---|---|
| 1 | | | | |
| 3 | | | | |
| 5 | | | | |
| 20 | | | | |



---
## Шаг 3. Построение амплитудного спектра

### Теория (кратко)

**Спектр** — это зависимость амплитуды гармоники от её частоты. Для периодического сигнала спектр **линейчатый** (дискретный): гармоники стоят на частотах $f_1, 2f_1, 3f_1, \dots$

Амплитуда $n$-й гармоники для нашей прямоугольной последовательности:
$$A_n = |a_n| = \left| \frac{2A}{n\pi}\sin\left(\frac{n\pi}{q}\right) \right|.$$

Постоянная составляющая: $A_0 = a_0/2 = A/q$.

**Важные свойства:**
- при $q = 2$ (меандр) все **чётные** гармоники равны нулю ($\sin(n\pi/2) = 0$ при чётном $n$);
- огибающая спектра убывает как $1/n$;
- чем **короче** импульс (больше $q$), тем **шире** спектр (больше гармоник с заметной амплитудой);
- постоянная составляющая $A/q$ уменьшается с ростом $q$.

### Что вы сейчас сделаете
1. Для $n = 0 \dots N$ посчитаете $A_n$ по формуле.
2. Построите **спектральную диаграмму** (stem-plot): по оси X — частота в кГц, по оси Y — амплитуда в В.
3. Сравните спектры для двух скважностей ($q = 2$ и $q = 4$).

**Ожидаемый результат:** «частокол» линий на частотах, кратных $f_1$; при $q=2$ чётные линии отсутствуют; при $q=4$ — каждая 4-я линия нулевая.


In [ ]:

# ============================================================
# ШАГ 3. АМПЛИТУДНЫЙ СПЕКТР ПРЯМОУГОЛЬНОГО СИГНАЛА
# ============================================================
N_spec = 20      # <--- ВПИШИТЕ СВОЕ ЗНАЧЕНИЕ ЗДЕСЬ: число гармоник для спектра

# ---------- НОМЕРА ГАРМОНИК И ИХ ЧАСТОТЫ ----------
n = np.arange(1, N_spec + 1)              # n = 1, 2, ..., N_spec
freq = n * f1                             # частоты гармоник, Гц

# ---------- АМПЛИТУДЫ ГАРМОНИК ----------
# An = |2A/(n*pi) * sin(n*pi/q)|
An = np.abs((2.0 * A / (n * np.pi)) * np.sin(n * np.pi / q))  # В

# ---------- ПОСТОЯННАЯ СОСТАВЛЯЮЩАЯ ----------
A0 = A / q                                # A0 = a0/2 = A/q, В

# ---------- СПЕКТРАЛЬНАЯ ДИАГРАММА ----------
plt.figure(figsize=(10, 4))
# stem — «частокол»: вертикальные линии из нуля до An
plt.stem(freq / 1000.0, An, basefmt='k-', linefmt='b-', markerfmt='bo')
plt.axhline(0, color='black', linewidth=0.8)       # ось нуля
plt.title(f'Амплитудный спектр: A={A} В, f1={f1} Гц, q={q:.2f}, N={N_spec}')
plt.xlabel('Частота f, кГц')
plt.ylabel('Амплитуда гармоники An, В')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xlim(0, (N_spec + 1) * f1 / 1000.0)
plt.tight_layout()
plt.show()

# ---------- ВЫВОД ПОСТОЯННОЙ СОСТАВЛЯЮЩЕЙ ----------
print(f'Постоянная составляющая A0 = A/q = {A0:.3f} В')
print(f'Первая гармоника A1 = {An[0]:.3f} В на частоте {f1/1000:.1f} кГц')


In [ ]:

# ============================================================
# ШАГ 3 (продолжение). СРАВНЕНИЕ СПЕКТРОВ ДЛЯ q=2 и q=4
# ============================================================
N_spec = 20
n = np.arange(1, N_spec + 1)
freq = n * f1

# Спектр для q = 2 (меандр, 50 %)
q1 = 2.0
An_q1 = np.abs((2.0 * A / (n * np.pi)) * np.sin(n * np.pi / q1))

# Спектр для q = 4 (25 %)
q2 = 4.0
An_q2 = np.abs((2.0 * A / (n * np.pi)) * np.sin(n * np.pi / q2))

plt.figure(figsize=(10, 4))
# Рисуем два спектра рядом: сдвигаем частоты на ±50 Гц, чтобы линии не наложились
plt.stem(freq / 1000.0 - 0.05, An_q1, basefmt='k-', linefmt='b-', markerfmt='bo', label='q = 2')
plt.stem(freq / 1000.0 + 0.05, An_q2, basefmt='k-', linefmt='r-', markerfmt='rs', label='q = 4')
plt.title('Сравнение амплитудных спектров при разной скважности')
plt.xlabel('Частота f, кГц')
plt.ylabel('Амплитуда гармоники An, В')
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.xlim(0, (N_spec + 1) * f1 / 1000.0)
plt.tight_layout()
plt.show()

print(f'Постоянная составляющая при q=2: A0 = {A/q1:.3f} В')
print(f'Постоянная составляющая при q=4: A0 = {A/q2:.3f} В')



### Параллельный эксперимент в NI Multisim (Шаг 3)

**Цель:** измерить амплитуды первых гармоник с помощью анализатора спектра `XSA1`.

**Что взять на схеме:**
| Компонент | Обозначение | Настройки |
|---|---|---|
| Функциональный генератор | `XFG1` | Прямоугольник, $f_1 = 1$ кГц, $A = 5$ В, Duty Cycle = 50 % |
| Анализатор спектра | `XSA1` | Вход — сигнал с `XFG1`; `Span` = 20 кГц; `Start` = 0 Гц; `Resolution` = 10 Гц |
| Земля | `GROUND` | Общая точка |

**Пошагово:**
1. Поставьте `XSA1` (группа `Instruments`) и подключите его вход к выходу `XFG1`.
2. Двойным щелчком откройте `XSA1`. Установите `Span` = 20 кГц (полоса обзора), `Start` = 0.
3. Нажмите **Run**. На экране анализатора появятся вертикальные пики на частотах $f_1, 3f_1, 5f_1, \dots$ (чётные должны отсутствовать!).
4. С помощью маркера (или курсора) считайте амплитуды первых трёх пиков. Занесите в таблицу.
5. Измените Duty Cycle на 25 % (это $q=4$), повторите измерения. Обратите внимание: появились 2-я и 4-я гармоники, а постоянная составляющая стала меньше.

> 💡 Если у `XSA1` нет выбора «прямоугольник», можно подать сигнал с `XFG1` — он есть в Multisim в группе `Sources` → `FUNCTION_GENERATOR`.

### Таблица сравнения результатов Шага 3

| Гармоника | Частота, кГц | $A_n$ теория (В) | $A_n$ Python (В) | $A_n$ Multisim (В) | Вывод |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |
| 5 | | | | | |
| $A_0$ (пост. сост.) | 0 | | | | |



---
## Шаг 4. Влияние скважности и постоянной составляющей

### Теория (кратко)

**Скважность** $q = T/\tau$ определяет:
- **постоянную составляющую** $A_0 = A/q$ — среднее значение сигнала;
- **огибающую спектра** $\propto \left|\sin(n\pi/q)\right|$ — какие гармоники «проваливаются» в ноль;
- **ширину спектра** — чем меньше $\tau$ (больше $q$), тем шире спектр.

**Постоянная составляющая** — это среднее значение сигнала за период:
$$A_0 = \frac{1}{T}\int_0^T s(t)\,dt.$$
Физически её можно измерить вольтметром постоянного тока (`DC_VOLTMETER`), она же соответствует «поднятию» всего спектра на нулевой частоте.

### Что вы сейчас сделаете
1. Зададите массив из нескольких значений $q$ (например, 2, 5, 10).
2. Для каждого $q$ построите график $s(t)$ и спектр (в одной фигуре — несколько подграфиков).
3. Убедитесь, что с ростом $q$ постоянная составляющая падает, а число заметных гармоник растёт.

**Ожидаемый результат:** семейство графиков, наглядно показывающее, как скважность «управляет» спектром.


In [ ]:

# ============================================================
# ШАГ 4. ВЛИЯНИЕ СКВАЖНОСТИ НА СПЕКТР И ПОСТОЯННУЮ СОСТАВЛЯЮЩУЮ
# ============================================================
q_list = [2.0, 5.0, 10.0]   # <--- МОЖЕТЕ ИЗМЕНИТЬ НАБОР ЗНАЧЕНИЙ q
N_spec = 30                 # число гармоник для спектра

fig, axes = plt.subplots(len(q_list), 2, figsize=(12, 3 * len(q_list)))
# axes — двумерный массив подграфиков: строка — вариант q, столбец — (время | спектр)

for i, q_val in enumerate(q_list):
    tau_val = T / q_val                          # длительность импульса для этого q
    # --- сигнал во времени ---
    s_t = np.where(np.mod(t, T) < tau_val, A, 0.0)
    axes[i, 0].plot(t * 1000, s_t, 'b', linewidth=1.5)
    axes[i, 0].set_title(f'Сигнал при q = {q_val:.0f} (τ = {tau_val*1e6:.0f} мкс)')
    axes[i, 0].set_xlabel('Время t, мс')
    axes[i, 0].set_ylabel('s(t), В')
    axes[i, 0].grid(True, linestyle='--', alpha=0.7)
    axes[i, 0].set_xlim(0, 2 * T * 1000)

    # --- спектр ---
    n = np.arange(1, N_spec + 1)
    An = np.abs((2.0 * A / (n * np.pi)) * np.sin(n * np.pi / q_val))
    axes[i, 1].stem(n * f1 / 1000.0, An, basefmt='k-', linefmt='r-', markerfmt='ro')
    axes[i, 1].set_title(f'Спектр при q = {q_val:.0f}: A0 = {A/q_val:.2f} В')
    axes[i, 1].set_xlabel('Частота f, кГц')
    axes[i, 1].set_ylabel('An, В')
    axes[i, 1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()



### Параллельный эксперимент в NI Multisim (Шаг 4)

**Цель:** измерить постоянную составляющую сигнала и убедиться, что она равна $A/q$.

**Что взять на схеме:**
| Компонент | Обозначение | Настройки |
|---|---|---|
| Функциональный генератор | `XFG1` | Прямоугольник, $f_1 = 1$ кГц, $A = 5$ В, Duty Cycle = 50 % (затем 20 %, 10 %) |
| Вольтметр постоянного тока | `DC_VOLTMETER` | Подключить параллельно выходу `XFG1` |
| Осциллограф | `XSC1` | Для контроля формы |

**Пошагово:**
1. Соберите схему: `XFG1` → `DC_VOLTMETER` (параллельно) → `XSC1` (канал A).
2. Задайте Duty Cycle = 50 % ($q=2$). Запустите **Run**. Запишите показание вольтметра (это $A_0$).
3. Повторите для Duty Cycle = 20 % ($q=5$) и 10 % ($q=10$). Занесите в таблицу.
4. Сравните с теоретическим $A_0 = A/q$: для $A = 5$ В и $q=2$ должно быть 2.5 В; для $q=5$ — 1.0 В; для $q=10$ — 0.5 В.
5. Дополнительно: откройте `XSA1` и убедитесь, что при $q=5$ «проваливаются» каждая 5-я гармоника ($n = 5, 10, 15, \dots$).

### Таблица сравнения результатов Шага 4

| Скважность $q$ | $\tau$, мкс | $A_0$ теория, В | $A_0$ Python, В | $A_0$ Multisim (DC-вольтметр), В | Вывод |
|---|---|---|---|---|---|
| 2 | | | | | |
| 5 | | | | | |
| 10 | | | | | |



---
## Таблица исходных данных по вариантам (10 вариантов)

| Вариант | Амплитуда $A$, В | Частота $f_1$, Гц | Длительность импульса $\tau$, мкс | Скважность $q = T/\tau$ |
|---|---|---|---|---|
| 1 | 1 | 1000 | 500 | 2 |
| 2 | 2 | 1000 | 250 | 4 |
| 3 | 3 | 1000 | 200 | 5 |
| 4 | 5 | 1000 | 500 | 2 |
| 5 | 5 | 2000 | 250 | 4 |
| 6 | 10 | 1000 | 100 | 10 |
| 7 | 4 | 2000 | 1000 | 2 |
| 8 | 6 | 500 | 125 | 8 |
| 9 | 8 | 1000 | 200 | 5 |
| 10 | 10 | 2000 | 500 | 2 |

> **Задание:** определите свой номер варианта у преподавателя, впишите соответствующие $A$, $f_1$, $\tau$ в ячейки Шагов 1–4 (помечены `# <--- ВПИШИТЕ СВОЕ ЗНАЧЕНИЕ ЗДЕСЬ`) и выполните все шаги.
